# 🎫 Support Tickets App - Lakebase Setup

This notebook sets up the Lakebase infrastructure required for the Support Tickets Dash application.

**What this notebook does:**
1. Creates a Lakebase database instance
2. Registers a Unity Catalog for the database
3. Seeds sample support ticket data

Run this notebook before deploying the Support Tickets app.

---
## Step 1: Configuration

Configure the Lakebase instance name and catalog. Update these values if needed for your environment.

In [ ]:
# Lakebase Configuration
LAKEBASE_INSTANCE_NAME = "support-tickets-lakebase"
LAKEBASE_CATALOG_NAME = "support_tickets_catalog"

# Schema and table used by the app
LAKEBASE_SCHEMA = "app"
LAKEBASE_TABLE = "support_tickets"

print(f"✅ Configuration loaded")
print(f"   Instance: {LAKEBASE_INSTANCE_NAME}")
print(f"   Catalog:  {LAKEBASE_CATALOG_NAME}")
print(f"   Table:    {LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}")

---
## Step 2: Create Lakebase Instance

A Lakebase instance is a Databricks-managed PostgreSQL database. Capacity units determine compute power:
- `CU_1`: Small workloads
- `CU_2`: Medium workloads (recommended for this demo)
- `CU_4`: Larger workloads

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import (
    DatabaseInstance, 
    DatabaseCatalog, 
    DatabaseInstanceState
)

# Initialize the Databricks workspace client
w = WorkspaceClient()

# Configure the database instance
db_instance_config = DatabaseInstance(
    name=LAKEBASE_INSTANCE_NAME,
    capacity="CU_2",  # Compute capacity: CU_1, CU_2, CU_4, etc.
)

# Create a new Lakebase database instance
try:
    instance = w.database.create_database_instance(db_instance_config)
    print(f"✅ Instance created: {db_instance_config.name}")
    print(f"   Please wait a few minutes for the instance to become available.")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"ℹ️  Instance '{db_instance_config.name}' already exists")
    else:
        print(f"❌ Error creating instance: {e}")

---
## Step 3: Wait for Instance to be Available

The Lakebase instance takes a few minutes to provision. Run this cell to check the status.

In [ ]:
import time

def wait_for_instance(instance_name, timeout_minutes=10):
    """Wait for the Lakebase instance to become available."""
    print(f"Waiting for instance '{instance_name}' to be available...")
    
    start_time = time.time()
    timeout_seconds = timeout_minutes * 60
    
    while True:
        try:
            instance = w.database.get_database_instance(instance_name)
            state = instance.state
            
            if state == DatabaseInstanceState.AVAILABLE:
                print(f"\n✅ Instance is AVAILABLE!")
                return True
            
            elapsed = int(time.time() - start_time)
            print(f"   Status: {state} (elapsed: {elapsed}s)", end="\r")
            
            if elapsed > timeout_seconds:
                print(f"\n⏰ Timeout after {timeout_minutes} minutes. Current state: {state}")
                return False
            
            time.sleep(10)
            
        except Exception as e:
            print(f"\n❌ Error checking instance: {e}")
            return False

# Wait for instance
instance_ready = wait_for_instance(LAKEBASE_INSTANCE_NAME)

---
## Step 4: Register Database Catalog

Register a Unity Catalog for the Lakebase instance. This allows you to manage the database through Unity Catalog.

In [ ]:
# Check if instance is available first
instance = w.database.get_database_instance(LAKEBASE_INSTANCE_NAME)

if instance.state != DatabaseInstanceState.AVAILABLE:
    print(f"⚠️  Instance is not ready yet. Current state: {instance.state}")
    print(f"   Please wait and run this cell again.")
else:
    # Configure the catalog
    catalog_config = DatabaseCatalog(
        name=LAKEBASE_CATALOG_NAME,
        database_instance_name=LAKEBASE_INSTANCE_NAME,
        database_name="databricks_postgres",  # Default Postgres database
    )
    
    # Register the catalog
    try:
        catalog = w.database.create_database_catalog(catalog_config)
        print(f"✅ Created database catalog: {catalog_config.name}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"ℹ️  Catalog '{catalog_config.name}' already exists")
        else:
            print(f"❌ Error creating catalog: {e}")

---
## Step 5: Create Schema and Table

Create the schema and support_tickets table that the app expects. We'll connect directly to the Lakebase instance using psycopg.

In [ ]:
# Get connection info for the Lakebase instance
instance = w.database.get_database_instance(LAKEBASE_INSTANCE_NAME)

# The connection details are available after the instance is created
# You'll use these when configuring the app.yml
print(f"📋 Lakebase Connection Info:")
print(f"   Instance Name: {LAKEBASE_INSTANCE_NAME}")
print(f"   Catalog Name:  {LAKEBASE_CATALOG_NAME}")
print(f"")
print(f"   Use these in your app.yml database resource configuration.")

In [ ]:
# Create schema and table using SQL through the catalog
# The app will auto-create the table on startup, but we can also create it here

create_schema_sql = f"CREATE SCHEMA IF NOT EXISTS {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}"

try:
    spark.sql(create_schema_sql)
    print(f"✅ Created schema: {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}")
except Exception as e:
    print(f"❌ Error creating schema: {e}")

In [ ]:
# Create the support_tickets table
create_table_sql = f"""
CREATE TABLE IF NOT EXISTS {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE} (
    id BIGINT GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
    title STRING NOT NULL,
    description STRING NOT NULL,
    customer_email STRING NOT NULL,
    status STRING NOT NULL DEFAULT 'open',
    priority STRING NOT NULL DEFAULT 'medium',
    created_at TIMESTAMP NOT NULL DEFAULT current_timestamp(),
    updated_at TIMESTAMP NOT NULL DEFAULT current_timestamp()
)
"""

try:
    spark.sql(create_table_sql)
    print(f"✅ Created table: {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}")
except Exception as e:
    print(f"❌ Error creating table: {e}")

---
## Step 6: Seed Sample Data

Insert sample support tickets so the app has data to display.

In [ ]:
from datetime import datetime, timedelta
import random

# Sample support ticket data
sample_tickets = [
    {
        "title": "Cannot login to dashboard",
        "description": "I keep getting an 'Invalid credentials' error when trying to log in. I've reset my password twice but still can't access my account.",
        "customer_email": "john.smith@acme.com",
        "status": "open",
        "priority": "high"
    },
    {
        "title": "Report export is slow",
        "description": "Exporting quarterly reports takes over 10 minutes. This used to complete in under a minute. Please investigate.",
        "customer_email": "sarah.johnson@techcorp.io",
        "status": "in_progress",
        "priority": "medium"
    },
    {
        "title": "Data sync not working",
        "description": "The data sync between our CRM and your platform stopped working yesterday. We're missing critical customer updates.",
        "customer_email": "mike.chen@enterprise.co",
        "status": "open",
        "priority": "critical"
    },
    {
        "title": "Feature request: Dark mode",
        "description": "Would love to have a dark mode option in the dashboard. It would be easier on the eyes during late night work sessions.",
        "customer_email": "lisa.wang@startup.io",
        "status": "open",
        "priority": "low"
    },
    {
        "title": "Billing discrepancy",
        "description": "I was charged twice for my monthly subscription. Please review and issue a refund for the duplicate charge.",
        "customer_email": "robert.taylor@business.net",
        "status": "resolved",
        "priority": "high"
    },
    {
        "title": "API rate limiting issues",
        "description": "We're hitting rate limits much sooner than expected based on our plan. Can you check if there's an issue with our quota?",
        "customer_email": "dev.team@api-users.com",
        "status": "in_progress",
        "priority": "medium"
    },
    {
        "title": "Mobile app crashes on startup",
        "description": "After the latest update, the iOS app crashes immediately upon opening. iPhone 14 Pro, iOS 17.2.",
        "customer_email": "emma.davis@mobile-user.com",
        "status": "open",
        "priority": "critical"
    },
    {
        "title": "Need help with integration",
        "description": "We're trying to integrate your API with our Salesforce instance. Documentation seems outdated. Can someone help?",
        "customer_email": "tech.support@salesforce-user.org",
        "status": "closed",
        "priority": "medium"
    },
    {
        "title": "Password reset email not received",
        "description": "Requested password reset 3 times, checked spam folder, still no email. Need access urgently for a client meeting.",
        "customer_email": "urgent.user@company.com",
        "status": "resolved",
        "priority": "high"
    },
    {
        "title": "Dashboard widgets not loading",
        "description": "The analytics widgets on my dashboard show 'Loading...' indefinitely. Tried clearing cache and different browsers.",
        "customer_email": "analytics.user@data.io",
        "status": "in_progress",
        "priority": "medium"
    }
]

print(f"📝 Sample tickets prepared: {len(sample_tickets)} tickets")

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import current_timestamp

# Create DataFrame from sample data
schema = StructType([
    StructField("title", StringType(), False),
    StructField("description", StringType(), False),
    StructField("customer_email", StringType(), False),
    StructField("status", StringType(), False),
    StructField("priority", StringType(), False)
])

# Convert to list of tuples for DataFrame creation
ticket_data = [
    (t["title"], t["description"], t["customer_email"], t["status"], t["priority"])
    for t in sample_tickets
]

df = spark.createDataFrame(ticket_data, schema)

# Add timestamp columns
df = df.withColumn("created_at", current_timestamp()) \
       .withColumn("updated_at", current_timestamp())

# Insert into table
try:
    df.write.mode("append").saveAsTable(f"{LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}")
    print(f"✅ Inserted {len(sample_tickets)} sample tickets into {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}")
except Exception as e:
    print(f"❌ Error inserting data: {e}")

---
## Step 7: Verify Data

Query the table to confirm the data was inserted correctly.

In [ ]:
# Verify the data
result_df = spark.sql(f"""
    SELECT id, title, status, priority, customer_email, created_at
    FROM {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}
    ORDER BY id
""")

print(f"📊 Support Tickets in Database:")
result_df.display()

In [ ]:
# Summary statistics
summary_df = spark.sql(f"""
    SELECT 
        status,
        COUNT(*) as count,
        COUNT(CASE WHEN priority = 'critical' THEN 1 END) as critical_count,
        COUNT(CASE WHEN priority = 'high' THEN 1 END) as high_count
    FROM {LAKEBASE_CATALOG_NAME}.{LAKEBASE_SCHEMA}.{LAKEBASE_TABLE}
    GROUP BY status
    ORDER BY status
""")

print(f"📈 Ticket Summary by Status:")
summary_df.display()

---
## ✅ Setup Complete!

Your Lakebase instance is ready. To deploy the Support Tickets app:

1. Update your `app.yml` to include the database resource:

```yaml
command:
  - python
  - app.py

resources:
  - name: support-tickets-db
    database:
      instance: support-tickets-lakebase
      catalog: support_tickets_catalog
```

2. Set environment variables (optional, these are the defaults):
   - `LAKEBASE_SCHEMA`: `app`
   - `LAKEBASE_TABLE`: `support_tickets`

3. Deploy the app using `databricks apps deploy`

---
## 🧹 Cleanup (Optional)

Run these cells to clean up resources if needed.

In [ ]:
# ⚠️ DANGER: Uncomment and run to delete all resources
# This will delete the catalog, instance, and all data!

# # Delete catalog first
# try:
#     w.database.delete_database_catalog(LAKEBASE_CATALOG_NAME)
#     print(f"🗑️ Deleted catalog: {LAKEBASE_CATALOG_NAME}")
# except Exception as e:
#     print(f"Error deleting catalog: {e}")

# # Then delete instance
# try:
#     w.database.delete_database_instance(LAKEBASE_INSTANCE_NAME)
#     print(f"🗑️ Deleted instance: {LAKEBASE_INSTANCE_NAME}")
# except Exception as e:
#     print(f"Error deleting instance: {e}")